# 09 - Debugging Gradients

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

from course.checks import qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 1. Debug 地图

| 现象 | 先检查 |
|---|---|
| 第二次运行 cell，grad 变成两倍 | 是否重复 backward 没清零 |
| loss 不下降 | update 方向、learning_rate、grad 是否为 0 |
| loss 震荡或变大 | learning_rate 可能太大 |
| 某些参数 grad 一直是 0 | ReLU 可能在负半轴断掉 |
| 训练越来越怪 | 是否每轮都 `zero_grad -> backward -> update` |

## 2. Bug 1：重复 backward 没清零

In [ ]:
w = Value(3.0)
L = w ** 2

L.backward()
print('after first backward :', w.grad)

L.backward()
print('after second backward:', w.grad)

print('原因：同一张图重复 backward，w.grad 会继续 += 新贡献。')

## 3. Bug 2：忘记 zero_grad

下面用一个标量训练任务：让 `w` 靠近 `1`。

$$
loss = (w - 1)^2
$$

In [ ]:
def train_scalar(reset_grad_each_step):
    w = Value(5.0)
    history = []
    for step in range(5):
        loss = (w - 1) ** 2
        history.append((w.data, loss.data, w.grad))

        if reset_grad_each_step:
            w.grad = 0
        loss.backward()
        w.data += -0.1 * w.grad
    return history, w.data

for reset in [True, False]:
    history, final_w = train_scalar(reset)
    print()
    print('reset_grad_each_step =', reset)
    for step, (w_data, loss_data, old_grad) in enumerate(history):
        print(f'step {step}: w={w_data:.4f}, loss={loss_data:.4f}, grad_before_backward={old_grad:.4f}')
    print('final w:', round(final_w, 4))

## 4. Bug 3：learning_rate 太大

还是：

$$
loss = w^2
$$

如果学习率太大，参数可能越过最低点，甚至越来越远。

In [ ]:
def train_with_lr(lr, steps=6):
    w = Value(5.0)
    values = []
    for _ in range(steps):
        w.grad = 0
        loss = w ** 2
        loss.backward()
        values.append((w.data, loss.data, w.grad))
        w.data += -lr * w.grad
    return values

for lr in [0.1, 0.6, 1.1]:
    print()
    print('lr =', lr)
    for w_data, loss_data, grad in train_with_lr(lr):
        print(f'w={w_data:>8.4f}, loss={loss_data:>8.4f}, grad={grad:>8.4f}')

## 5. Bug 4：ReLU 断掉

ReLU 在负数区域导数是 0：

```text
x < 0 -> relu(x) = 0 -> grad = 0
```

所以如果某个神经元一直在负半轴，它这次不会把梯度传回去。

In [ ]:
for value in [-2.0, 3.0]:
    x = Value(value)
    y = x.relu()
    y.backward()
    print(f'x={value:>4}, relu={y.data:>4}, x.grad={x.grad}')

## What To Remember

```text
1. grad 会累加，所以每轮 backward 前要清零。
2. 同一张计算图重复 backward，grad 可能变成两倍、三倍。
3. learning_rate 太小下降慢，太大可能震荡或变差。
4. ReLU 在负数区域不传梯度。
5. 调试顺序先看：loss、grad、zero_grad、update、learning_rate。
```

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - grad 为什么翻倍？

In [ ]:
# 填写说明：
# - 填一句中文：解释重复 backward 或不清零时 grad 为什么会累加。
# - 填完后运行本 cell，看 qa_check 的提示。

student_reason = None



qa_check('qa_check_class_09_predict', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先看症状再找原因：grad 变大查 zero_grad，loss 变大查更新方向，参数不动查 update。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- 答案需要满足：`'累加' in student_reason or 'zero_grad' in student_reason`。

</details>

## Run - 重复 backward 的症状

In [ ]:
from micrograd.engine import Value

w = Value(2.0)
L = w*3
L.backward()
print('first:', w.grad)
L.backward()
print('second:', w.grad)

## 拆开看 - 常见症状表

In [ ]:
cases = [
    ('grad 一直累加', '忘记 zero_grad'),
    ('loss 稳定变大', '更新方向反了'),
    ('loss 爆炸跳来跳去', 'learning_rate 太大'),
    ('参数完全不动', '漏了 update'),
    ('ReLU 后 grad 全 0', 'dead ReLU'),
]
for symptom, reason in cases:
    print(f'{symptom:18s} -> {reason}')

## Modify - 修更新方向

In [ ]:
# 填写说明：
# - 填字符串：修正后的参数更新代码，格式像 p.data = ...。
# - 填完后运行本 cell，看 qa_check 的提示。

student_fixed_update = None  # 'p.data = p.data - lr * p.grad'



qa_check('qa_check_class_09_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先看症状再找原因：grad 变大查 zero_grad，loss 变大查更新方向，参数不动查 update。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- 答案需要满足：`student_fixed_update.replace(' '`。

</details>